# Baseline XResNet1d50 — same train rows as balanced-sex SSSD generator

Trains **XResNet1d50** on **real PTB-XL only** using the **exact train row indices** saved by `Train_custom_SSSD_ECG_balanced_sex.ipynb`.

- Train rows: `generator_train_row_indices_seed{SUBSET_SEED}.npy` from balanced subset Drive folder.
- Val/test rows: standard PTB-XL fold split (same as other notebooks).
- Target: binary **MI** label derived from the same SCP rule.

This notebook intentionally skips synthetic-mixture experiments for now.

In [ ]:
import sys, torch, numpy as np, scipy
print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("NumPy:", np.__version__)
print("SciPy:", scipy.__version__)

In [ ]:
!pip install -q pykeops
!pip install -q wfdb resampy ishneholterlib pytorch-lightning scikit-learn

In [ ]:
import os
import sys
from pathlib import Path

REPO_URL = "https://github.com/Anonymous0002396/CMED-Mini-Project.git"
PROJECT_ROOT = Path('/content/CMED-Mini-Project')
REPO_ROOT = PROJECT_ROOT / 'SSSD-ECG'

if not PROJECT_ROOT.exists():
    !git clone {REPO_URL} /content/CMED-Mini-Project
else:
    %cd /content/CMED-Mini-Project
    !git pull

sys.path.insert(0, str(REPO_ROOT / 'src'))
sys.path.insert(0, str(REPO_ROOT / 'src' / 'ptb_xl'))
sys.path.insert(0, str(REPO_ROOT / 'src' / 'sssd'))

print('REPO_ROOT:', REPO_ROOT)
print('train.py exists:', (REPO_ROOT / 'src' / 'sssd' / 'train.py').exists())

In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')

if not os.path.exists('/content/ptbxl.zip'):
    !cp "/content/drive/MyDrive/ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.zip" /content/ptbxl.zip

if not os.path.exists('/content/ptb-xl-a-large-publicly-available-electrocardiography-dataset-1'):
    !unzip -oq /content/ptbxl.zip -d /content/
    print('Extraction complete.')
else:
    print('PTB-XL already extracted, skipping extraction.')

raw_ptbxl = Path('/content/ptb-xl-a-large-publicly-available-electrocardiography-dataset-1')
print('raw_ptbxl exists:', raw_ptbxl.exists())

In [ ]:
import numpy as np
import pandas as pd
import pickle
from pathlib import Path

from clinical_ts.ecg_utils import prepare_data_ptb_xl, channel_stoi_default
from clinical_ts.timeseries_utils import reformat_as_memmap, load_dataset

# Preprocess PTB-XL at 100 Hz (same family as training notebooks)
target_fs = 100
data_folder_ptb_xl = Path('/content/ptb-xl-a-large-publicly-available-electrocardiography-dataset-1')
target_folder_ptb_xl = Path(f'/content/processed_ptb_xl_fs{target_fs}')

if target_folder_ptb_xl.exists():
    !rm -rf {target_folder_ptb_xl}

print('Rebuilding processed PTB-XL at:', target_folder_ptb_xl)

df_ptb_xl, lbl_itos_ptb_xl, mean_ptb_xl, std_ptb_xl = prepare_data_ptb_xl(
    data_path=data_folder_ptb_xl,
    min_cnt=0,
    target_fs=target_fs,
    channels=12,
    channel_stoi=channel_stoi_default,
    target_folder=target_folder_ptb_xl,
    recreate_data=True,
)

reformat_as_memmap(
    df_ptb_xl,
    target_folder_ptb_xl / 'memmap.npy',
    data_folder=target_folder_ptb_xl,
    delete_npys=True,
)

df_mapped, lbl_itos_dict, mean, std = load_dataset(target_folder_ptb_xl)
print('Processed df shape:', df_mapped.shape)

# Build MI labels from 71 SCP multihot
label_key = 'label_all'
label_names = np.array(lbl_itos_dict[label_key])

mi_statement_names = [
    'IMI', 'ASMI', 'ILMI', 'AMI', 'ALMI',
    'INJAS', 'LMI', 'INJAL', 'IPLMI', 'IPMI',
    'INJIN', 'PMI', 'INJLA', 'INJIL'
]
label_name_to_idx = {str(name): i for i, name in enumerate(label_names)}
mi_label_indices = [label_name_to_idx[name] for name in mi_statement_names]

def multihot_encode(indices, num_classes):
    out = np.zeros(num_classes, dtype=np.float32)
    for i in indices:
        out[i] = 1.0
    return out

def row_multihot_to_binary_mi(row_vec, mi_indices):
    return float(row_vec[mi_indices].sum() > 0)

df_real = df_mapped.copy()
df_real['label_multihot'] = df_real[f'{label_key}_numeric'].apply(lambda x: multihot_encode(x, len(label_names)))
df_real['label_mi'] = df_real['label_multihot'].apply(lambda x: row_multihot_to_binary_mi(x, mi_label_indices)).astype(np.float32)
df_real['sex_binary'] = df_real['sex'].astype(np.float32)

with open(target_folder_ptb_xl / 'df_memmap.pkl', 'rb') as f:
    df_memmap_meta = pickle.load(f)

df_memmap_meta = df_memmap_meta.copy()
df_memmap_meta['sex_binary'] = df_real['sex_binary'].values.astype(np.float32)
df_memmap_meta['label_mi'] = df_real['label_mi'].values.astype(np.float32)
df_memmap_meta['strat_fold'] = df_real['strat_fold'].values

print(df_memmap_meta.groupby(['sex_binary', 'label_mi']).size())

In [ ]:
# Match Train_custom_SSSD_ECG_balanced_sex.ipynb subset rows
SUBSET_SEED = 42
DRIVE_SUBSET_DIR = Path('/content/drive/MyDrive/sssd_sex_scp71_balanced_50f_50m_subsets')

idx_path = DRIVE_SUBSET_DIR / f'generator_train_row_indices_seed{SUBSET_SEED}.npy'
if not idx_path.exists():
    raise FileNotFoundError(
        f'Missing {idx_path}. Run the balanced SSSD notebook cell that saves indices, '
        'or set SUBSET_SEED / DRIVE_SUBSET_DIR to your run.'
    )

row_indices = np.load(idx_path)
missing = np.setdiff1d(row_indices, df_memmap_meta.index.to_numpy())
if len(missing):
    raise ValueError(
        f'{len(missing)} indices from {idx_path.name} are not in df_memmap_meta — '
        'raw/preprocessed PTB-XL likely differs from the subset run.'
    )

max_fold_id = int(df_memmap_meta['strat_fold'].max())
df_val_real_mem = df_memmap_meta[df_memmap_meta['strat_fold'] == max_fold_id - 1].copy()
df_test_real_mem = df_memmap_meta[df_memmap_meta['strat_fold'] == max_fold_id].copy()

df_train_real_mem_subset = df_memmap_meta.loc[row_indices].copy()

print('Train subset:', len(df_train_real_mem_subset))
print('Val:', len(df_val_real_mem), '| Test:', len(df_test_real_mem))
print('Female fraction (train):', (df_train_real_mem_subset['sex_binary'] == 1.0).mean())
print(df_train_real_mem_subset.groupby(['sex_binary', 'label_mi']).size())

In [ ]:
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from clinical_ts.timeseries_utils import TimeseriesDatasetCrops, ToTensor

input_size = 1000
tfms_ptb_xl = ToTensor()

# For classifier, label comes directly from label_mi (scalar)
train_real_ds_full12_subset = TimeseriesDatasetCrops(
    df_train_real_mem_subset,
    output_size=input_size,
    data_folder=target_folder_ptb_xl,
    chunk_length=0,
    min_chunk_length=input_size,
    stride=input_size,
    transforms=tfms_ptb_xl,
    annotation=False,
    col_lbl='label_mi',
    memmap_filename=target_folder_ptb_xl / 'memmap.npy',
)

val_real_ds_full12 = TimeseriesDatasetCrops(
    df_val_real_mem,
    output_size=input_size,
    data_folder=target_folder_ptb_xl,
    chunk_length=0,
    min_chunk_length=input_size,
    stride=input_size,
    transforms=tfms_ptb_xl,
    annotation=False,
    col_lbl='label_mi',
    memmap_filename=target_folder_ptb_xl / 'memmap.npy',
)

test_real_ds_full12 = TimeseriesDatasetCrops(
    df_test_real_mem,
    output_size=input_size,
    data_folder=target_folder_ptb_xl,
    chunk_length=0,
    min_chunk_length=input_size,
    stride=input_size,
    transforms=tfms_ptb_xl,
    annotation=False,
    col_lbl='label_mi',
    memmap_filename=target_folder_ptb_xl / 'memmap.npy',
)


class TupleToDictWrapper(Dataset):
    def __init__(self, base_ds):
        self.base_ds = base_ds

    def __len__(self):
        return len(self.base_ds)

    def __getitem__(self, idx):
        x, y = self.base_ds[idx]
        x = x.float() if torch.is_tensor(x) else torch.tensor(x, dtype=torch.float32)
        y = y.float() if torch.is_tensor(y) else torch.tensor(y, dtype=torch.float32)
        return {
            'signal': x,
            'label': y,
        }


train_real_ds = TupleToDictWrapper(train_real_ds_full12_subset)
val_real_ds = TupleToDictWrapper(val_real_ds_full12)
test_real_ds = TupleToDictWrapper(test_real_ds_full12)

batch_size = 64

def make_train_real_loader(train_seed: int) -> DataLoader:
    g = torch.Generator()
    g.manual_seed(int(train_seed))
    return DataLoader(
        train_real_ds,
        batch_size=batch_size,
        shuffle=True,
        num_workers=2,
        pin_memory=True,
        generator=g,
    )

val_loader = DataLoader(val_real_ds, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_real_ds, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)

print('train:', len(train_real_ds), 'val:', len(val_real_ds), 'test:', len(test_real_ds))

In [ ]:
import random
import numpy as np
import torch
import torch.nn as nn
from sklearn.metrics import roc_auc_score, average_precision_score


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
criterion = nn.BCEWithLogitsLoss()


def set_train_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def train_one_epoch(model, loader, optimizer, device):
    model.train()
    total_loss, n = 0.0, 0
    for batch in loader:
        x = batch['signal'].to(device, non_blocking=True)
        y = batch['label'].to(device, non_blocking=True).float()
        optimizer.zero_grad()
        logits = model(x)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()
        bs = x.size(0)
        total_loss += loss.item() * bs
        n += bs
    return total_loss / max(n, 1)


@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()
    total_loss, n = 0.0, 0
    all_probs, all_targets = [], []
    for batch in loader:
        x = batch['signal'].to(device, non_blocking=True)
        y = batch['label'].to(device, non_blocking=True).float()
        logits = model(x)
        loss = criterion(logits, y)
        probs = torch.sigmoid(logits)
        bs = x.size(0)
        total_loss += loss.item() * bs
        n += bs
        all_probs.append(probs.cpu())
        all_targets.append(y.cpu())
    all_probs = torch.cat(all_probs).numpy()
    all_targets = torch.cat(all_targets).numpy()
    return {
        'loss': total_loss / max(n, 1),
        'auroc': roc_auc_score(all_targets, all_probs),
        'auprc': average_precision_score(all_targets, all_probs),
        'probs': all_probs,
        'targets': all_targets,
    }


def fit_model(model, train_loader, val_loader, device, epochs=20, lr=1e-3, verbose=True):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    best_state = None
    best_val_auroc = -np.inf
    for epoch in range(1, epochs + 1):
        train_loss = train_one_epoch(model, train_loader, optimizer, device)
        val_metrics = evaluate(model, val_loader, device)
        if verbose:
            print(
                f"Epoch {epoch}/{epochs} | train_loss={train_loss:.4f} | "
                f"val_loss={val_metrics['loss']:.4f} | val_auroc={val_metrics['auroc']:.4f} | "
                f"val_auprc={val_metrics['auprc']:.4f}"
            )
        if val_metrics['auroc'] > best_val_auroc:
            best_val_auroc = val_metrics['auroc']
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
    model.load_state_dict(best_state)
    return model


def subgroup_metrics(y_true, y_prob, mask):
    y_t = y_true[mask]
    y_p = y_prob[mask]
    return {
        'auroc': roc_auc_score(y_t, y_p),
        'auprc': average_precision_score(y_t, y_p),
        'n': int(mask.sum()),
    }

female_test_mask = df_test_real_mem['sex_binary'].to_numpy() == 1.0
male_test_mask = df_test_real_mem['sex_binary'].to_numpy() == 0.0

In [ ]:
import torch
import torch.nn as nn


class Bottleneck1D(nn.Module):
    expansion = 4

    def __init__(self, in_channels, planes, stride=1):
        super().__init__()
        out_channels = planes * self.expansion
        self.conv1 = nn.Conv1d(in_channels, planes, kernel_size=1, bias=False)
        self.bn1 = nn.BatchNorm1d(planes)
        self.conv2 = nn.Conv1d(planes, planes, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn2 = nn.BatchNorm1d(planes)
        self.conv3 = nn.Conv1d(planes, out_channels, kernel_size=1, bias=False)
        self.bn3 = nn.BatchNorm1d(out_channels)
        self.relu = nn.ReLU(inplace=True)
        self.downsample = None
        if stride != 1 or in_channels != out_channels:
            self.downsample = nn.Sequential(
                nn.Conv1d(in_channels, out_channels, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm1d(out_channels),
            )

    def forward(self, x):
        identity = x
        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)
        out = self.conv2(out)
        out = self.bn2(out)
        out = self.relu(out)
        out = self.conv3(out)
        out = self.bn3(out)
        if self.downsample is not None:
            identity = self.downsample(x)
        out = out + identity
        out = self.relu(out)
        return out


class XResNet1d50BinaryMIClassifier(nn.Module):
    def __init__(self, in_channels=12, base_filters=64, num_classes=1):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv1d(in_channels, 32, kernel_size=5, stride=2, padding=2, bias=False),
            nn.BatchNorm1d(32),
            nn.ReLU(inplace=True),
            nn.Conv1d(32, 32, kernel_size=3, stride=1, padding=1, bias=False),
            nn.BatchNorm1d(32),
            nn.ReLU(inplace=True),
            nn.Conv1d(32, base_filters, kernel_size=3, stride=1, padding=1, bias=False),
            nn.BatchNorm1d(base_filters),
            nn.ReLU(inplace=True),
        )
        self.in_channels = base_filters
        self.layer1 = self._make_layer(64, 3, stride=1)
        self.layer2 = self._make_layer(128, 4, stride=2)
        self.layer3 = self._make_layer(256, 6, stride=2)
        self.layer4 = self._make_layer(512, 3, stride=2)
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.fc = nn.Linear(512 * Bottleneck1D.expansion, num_classes)

    def _make_layer(self, planes, blocks, stride):
        layers = [Bottleneck1D(self.in_channels, planes, stride=stride)]
        self.in_channels = planes * Bottleneck1D.expansion
        for _ in range(1, blocks):
            layers.append(Bottleneck1D(self.in_channels, planes, stride=1))
        return nn.Sequential(*layers)

    def forward(self, signal):
        x = self.stem(signal)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.pool(x).squeeze(-1)
        logits = self.fc(x).squeeze(1)
        return logits

In [ ]:
# Baseline: real-only XResNet on balanced-generator train subset
import numpy as np
import pandas as pd

TRAIN_SEEDS = [0, 1, 2, 3,4]
EPOCHS = 100
LR = 1e-3
VERBOSE_EPOCHS = False

seed_results = []
xresnet = None

for train_seed in TRAIN_SEEDS:
    print(f"\n========== TRAIN_SEED {train_seed} ==========")
    set_train_seed(train_seed)

    train_real_loader = make_train_real_loader(train_seed)
    xresnet = XResNet1d50BinaryMIClassifier(in_channels=12).to(device)
    xresnet = fit_model(
        xresnet,
        train_real_loader,
        val_loader,
        device,
        epochs=EPOCHS,
        lr=LR,
        verbose=VERBOSE_EPOCHS,
    )

    test_metrics = evaluate(xresnet, test_loader, device)
    female_metrics = subgroup_metrics(test_metrics['targets'], test_metrics['probs'], female_test_mask)
    male_metrics = subgroup_metrics(test_metrics['targets'], test_metrics['probs'], male_test_mask)

    print(
        f"TEST train_seed={train_seed} | AUROC={test_metrics['auroc']:.4f} "
        f"AUPRC={test_metrics['auprc']:.4f} | "
        f"F_AUROC={female_metrics['auroc']:.4f} M_AUROC={male_metrics['auroc']:.4f}"
    )

    seed_results.append(
        {
            'train_seed': int(train_seed),
            'test_loss': float(test_metrics['loss']),
            'test_auroc': float(test_metrics['auroc']),
            'test_auprc': float(test_metrics['auprc']),
            'female_auroc': float(female_metrics['auroc']),
            'female_auprc': float(female_metrics['auprc']),
            'male_auroc': float(male_metrics['auroc']),
            'male_auprc': float(male_metrics['auprc']),
            'female_n': int(female_metrics['n']),
            'male_n': int(male_metrics['n']),
        }
    )

summary = {
    'subset_seed': int(SUBSET_SEED),
    'n_train_subset': int(len(train_real_ds)),
    'epochs': int(EPOCHS),
    'lr': float(LR),
    'seeds': TRAIN_SEEDS,
}
for key in ['test_auroc', 'test_auprc', 'female_auroc', 'female_auprc', 'male_auroc', 'male_auprc']:
    a = np.array([r[key] for r in seed_results], dtype=np.float64)
    summary[f'{key}_mean'] = float(a.mean())
    summary[f'{key}_std'] = float(a.std(ddof=1)) if len(a) > 1 else 0.0

print('\n========== BASELINE aggregate over seeds ==========')
for key in ['test_auroc', 'test_auprc', 'female_auroc', 'female_auprc', 'male_auroc', 'male_auprc']:
    print(f"  {key}: {summary[key + '_mean']:.4f} ± {summary[key + '_std']:.4f}")

display(pd.DataFrame(seed_results))
print('\nSummary dict:')
print(summary)

## Real + Synthetic (Direct Concatenation, no mixing)

This section appends synthetic ECGs directly to the real train subset and trains the same XResNet1d50 MI classifier. It does **not** use probabilistic alpha-mixing. Training is run for **75 epochs**.

In [ ]:
import numpy as np
import torch
from pathlib import Path
from torch.utils.data import Dataset, DataLoader, ConcatDataset
import pandas as pd

# Synthetic data produced by Generate_ECGs_on_balanced_sex.ipynb
SYN_SAVE_DIR = Path('/content/drive/MyDrive/sex_scp71_synthetic_balanced_female_mi3x_ckpt20000')

syn_data_path = SYN_SAVE_DIR / 'syn_data_12.npy'
syn_cond_path = SYN_SAVE_DIR / 'syn_cond_72.npy'
if not syn_data_path.is_file() or not syn_cond_path.is_file():
    raise FileNotFoundError(
        f'Missing synthetic files at {SYN_SAVE_DIR}. Expected syn_data_12.npy and syn_cond_72.npy.'
    )

syn_data = np.load(syn_data_path).astype(np.float32)
syn_cond = np.load(syn_cond_path).astype(np.float32)

# Robust MI extraction from synthetic conditioning
# - If cond is [sex, mi], MI is col 1
# - If cond is [sex + 71 labels], derive MI from MI SCP indices in the 71-label block
if syn_cond.shape[1] == 2:
    syn_mi = syn_cond[:, 1].astype(np.float32)
elif syn_cond.shape[1] >= 72:
    mi_cols = [1 + int(i) for i in mi_label_indices]  # offset by sex channel
    syn_mi = (syn_cond[:, mi_cols].sum(axis=1) > 0).astype(np.float32)
else:
    raise ValueError(f'Unexpected synthetic cond shape: {syn_cond.shape}')


class SyntheticECGMIDataset(Dataset):
    def __init__(self, signals, mi_labels):
        self.signals = np.asarray(signals, dtype=np.float32)
        self.labels = np.asarray(mi_labels, dtype=np.float32)

    def __len__(self):
        return len(self.signals)

    def __getitem__(self, idx):
        return {
            "signal": torch.tensor(self.signals[idx], dtype=torch.float32),
            "label": torch.tensor(self.labels[idx], dtype=torch.float32),
        }


train_syn_ds = SyntheticECGMIDataset(syn_data, syn_mi)
train_aug_ds = ConcatDataset([train_real_ds, train_syn_ds])


def make_train_aug_loader(train_seed: int) -> DataLoader:
    g = torch.Generator()
    g.manual_seed(int(train_seed))
    return DataLoader(
        train_aug_ds,
        batch_size=batch_size,
        shuffle=True,
        num_workers=2,
        pin_memory=True,
        generator=g,
    )


print('Real train size:', len(train_real_ds))
print('Synthetic train size:', len(train_syn_ds))
print('Augmented train size:', len(train_aug_ds))
print('Synthetic MI counts:', np.unique(syn_mi, return_counts=True))

# Train augmented model for 75 epochs (as requested)
TRAIN_SEEDS_AUG = [0, 1, 2, 3, 4]
EPOCHS_AUG = 75
LR_AUG = 1e-3
VERBOSE_EPOCHS_AUG = False

aug_results = []
xresnet_aug = None

for train_seed in TRAIN_SEEDS_AUG:
    print(f"\n========== AUG TRAIN_SEED {train_seed} ==========")
    set_train_seed(train_seed)

    train_aug_loader = make_train_aug_loader(train_seed)
    xresnet_aug = XResNet1d50BinaryMIClassifier(in_channels=12).to(device)
    xresnet_aug = fit_model(
        xresnet_aug,
        train_aug_loader,
        val_loader,
        device,
        epochs=EPOCHS_AUG,
        lr=LR_AUG,
        verbose=VERBOSE_EPOCHS_AUG,
    )

    test_metrics = evaluate(xresnet_aug, test_loader, device)
    female_metrics = subgroup_metrics(test_metrics['targets'], test_metrics['probs'], female_test_mask)
    male_metrics = subgroup_metrics(test_metrics['targets'], test_metrics['probs'], male_test_mask)

    print(
        f"AUG TEST seed={train_seed} | AUROC={test_metrics['auroc']:.4f} "
        f"AUPRC={test_metrics['auprc']:.4f} | "
        f"F_AUROC={female_metrics['auroc']:.4f} M_AUROC={male_metrics['auroc']:.4f}"
    )

    aug_results.append(
        {
            'train_seed': int(train_seed),
            'test_loss': float(test_metrics['loss']),
            'test_auroc': float(test_metrics['auroc']),
            'test_auprc': float(test_metrics['auprc']),
            'female_auroc': float(female_metrics['auroc']),
            'female_auprc': float(female_metrics['auprc']),
            'male_auroc': float(male_metrics['auroc']),
            'male_auprc': float(male_metrics['auprc']),
        }
    )

aug_summary = {
    'model': 'XResNet1d50BinaryMIClassifier',
    'setup': 'real + synthetic direct concat',
    'epochs': int(EPOCHS_AUG),
    'lr': float(LR_AUG),
    'seeds': TRAIN_SEEDS_AUG,
}
for key in ['test_auroc', 'test_auprc', 'female_auroc', 'female_auprc', 'male_auroc', 'male_auprc']:
    a = np.array([r[key] for r in aug_results], dtype=np.float64)
    aug_summary[f'{key}_mean'] = float(a.mean())
    aug_summary[f'{key}_std'] = float(a.std(ddof=1)) if len(a) > 1 else 0.0

print('\n========== AUGMENTED aggregate over seeds ==========')
for key in ['test_auroc', 'test_auprc', 'female_auroc', 'female_auprc', 'male_auroc', 'male_auprc']:
    print(f"  {key}: {aug_summary[key + '_mean']:.4f} ± {aug_summary[key + '_std']:.4f}")

display(pd.DataFrame(aug_results))

# Optional side-by-side with baseline summary from previous section
if 'summary' in globals():
    compare_rows = []
    for metric in ['test_auroc', 'test_auprc', 'female_auroc', 'female_auprc', 'male_auroc', 'male_auprc']:
        b = summary.get(f'{metric}_mean', np.nan)
        a = aug_summary.get(f'{metric}_mean', np.nan)
        compare_rows.append({'metric': metric, 'baseline_mean': b, 'aug_mean': a, 'delta_aug_minus_base': a - b})
    print('\nBaseline vs Augmented (mean over seeds):')
    display(pd.DataFrame(compare_rows))

## Real + Synthetic Mixing (alpha = 0.5, without replacement)

This section uses weighted sampling over the concatenated real+synthetic training dataset with **`alpha = 0.5`** (target probability of drawing real samples) and **sampling without replacement**.

In [ ]:
import numpy as np
import torch
from torch.utils.data import Dataset, ConcatDataset, WeightedRandomSampler, DataLoader
import pandas as pd

# Reuse synthetic dataset from previous section if available; otherwise load it.
if 'train_syn_ds' not in globals():
    SYN_SAVE_DIR = Path('/content/drive/MyDrive/sex_scp71_synthetic_balanced_female_mi3x_ckpt20000')
    syn_data = np.load(SYN_SAVE_DIR / 'syn_data_12.npy').astype(np.float32)
    syn_cond = np.load(SYN_SAVE_DIR / 'syn_cond_72.npy').astype(np.float32)

    if syn_cond.shape[1] == 2:
        syn_mi = syn_cond[:, 1].astype(np.float32)
    elif syn_cond.shape[1] >= 72:
        mi_cols = [1 + int(i) for i in mi_label_indices]
        syn_mi = (syn_cond[:, mi_cols].sum(axis=1) > 0).astype(np.float32)
    else:
        raise ValueError(f'Unexpected synthetic cond shape: {syn_cond.shape}')

    class SyntheticECGMIDataset(Dataset):
        def __init__(self, signals, mi_labels):
            self.signals = np.asarray(signals, dtype=np.float32)
            self.labels = np.asarray(mi_labels, dtype=np.float32)

        def __len__(self):
            return len(self.signals)

        def __getitem__(self, idx):
            return {
                'signal': torch.tensor(self.signals[idx], dtype=torch.float32),
                'label': torch.tensor(self.labels[idx], dtype=torch.float32),
            }

    train_syn_ds = SyntheticECGMIDataset(syn_data, syn_mi)


# Mixing config
ALPHA = 0.5
MIX_TRAIN_SEEDS = [0, 1, 2, 3, 4]
EPOCHS_MIX = 100
LR_MIX = 1e-3
VERBOSE_MIX_EPOCHS = False

R = len(train_real_ds)
S = len(train_syn_ds)
num_samples_per_epoch = R  # keep batches/epoch aligned with baseline
baseline_batches = (R + batch_size - 1) // batch_size

print('real train size:', R)
print('synthetic train size:', S)
print('alpha:', ALPHA)
print('num_samples_per_epoch:', num_samples_per_epoch)


def make_mix_train_loader(alpha: float, train_seed: int) -> DataLoader:
    eps = 1e-10
    a = float(np.clip(alpha, eps, 1.0 - eps))

    combined = ConcatDataset([train_real_ds, train_syn_ds])

    # Expected mass: a on real block, (1-a) on synthetic block
    w = np.concatenate(
        [
            np.full(R, (a / R), dtype=np.float64),
            np.full(S, ((1.0 - a) / S), dtype=np.float64),
        ]
    )

    g = torch.Generator()
    g.manual_seed(int(train_seed))

    sampler = WeightedRandomSampler(
        weights=torch.as_tensor(w, dtype=torch.double),
        num_samples=int(num_samples_per_epoch),
        replacement=False,  # requested: WITHOUT replacement
        generator=g,
    )

    loader = DataLoader(
        combined,
        batch_size=batch_size,
        sampler=sampler,
        num_workers=2,
        pin_memory=True,
    )

    if len(loader) != baseline_batches:
        raise RuntimeError(
            f'Batch count mismatch: mix len(loader)={len(loader)} vs baseline {baseline_batches}'
        )

    return loader


mix_results = []
xresnet_mix = None

for mix_seed in MIX_TRAIN_SEEDS:
    print(f"\n========== MIX seed={mix_seed} (alpha={ALPHA}) ==========")
    set_train_seed(mix_seed)

    train_mix_loader = make_mix_train_loader(ALPHA, mix_seed)

    xresnet_mix = XResNet1d50BinaryMIClassifier(in_channels=12).to(device)
    xresnet_mix = fit_model(
        xresnet_mix,
        train_mix_loader,
        val_loader,
        device,
        epochs=EPOCHS_MIX,
        lr=LR_MIX,
        verbose=VERBOSE_MIX_EPOCHS,
    )

    tm = evaluate(xresnet_mix, test_loader, device)
    fem = subgroup_metrics(tm['targets'], tm['probs'], female_test_mask)
    mal = subgroup_metrics(tm['targets'], tm['probs'], male_test_mask)

    row = {
        'alpha': float(ALPHA),
        'mix_seed': int(mix_seed),
        'epochs': int(EPOCHS_MIX),
        'samples_per_epoch': int(num_samples_per_epoch),
        'batches_per_epoch': int(baseline_batches),
        'test_loss': float(tm['loss']),
        'test_auroc': float(tm['auroc']),
        'test_auprc': float(tm['auprc']),
        'female_auroc': float(fem['auroc']),
        'female_auprc': float(fem['auprc']),
        'male_auroc': float(mal['auroc']),
        'male_auprc': float(mal['auprc']),
    }
    mix_results.append(row)

    print(
        f"TEST | AUROC={row['test_auroc']:.4f} AUPRC={row['test_auprc']:.4f} | "
        f"F_AUROC={row['female_auroc']:.4f} M_AUROC={row['male_auroc']:.4f}"
    )


mix_summary = {'alpha': float(ALPHA), 'n_seeds': len(mix_results)}
for key in ['test_auroc', 'test_auprc', 'female_auroc', 'female_auprc', 'male_auroc', 'male_auprc']:
    a = np.array([r[key] for r in mix_results], dtype=np.float64)
    mix_summary[f'{key}_mean'] = float(a.mean())
    mix_summary[f'{key}_std'] = float(a.std(ddof=1)) if len(a) > 1 else 0.0

print('\n========== MIX aggregate over seeds (alpha=0.5) ==========')
for key in ['test_auroc', 'test_auprc', 'female_auroc', 'female_auprc', 'male_auroc', 'male_auprc']:
    print(f"  {key}: {mix_summary[key + '_mean']:.4f} ± {mix_summary[key + '_std']:.4f}")

display(pd.DataFrame(mix_results))

if 'summary' in globals():
    print('\nBaseline (real-only) mean vs Mix(alpha=0.5) mean:')
    rows = []
    for metric in ['test_auroc', 'test_auprc', 'female_auroc', 'female_auprc', 'male_auroc', 'male_auprc']:
        b = summary.get(f'{metric}_mean', np.nan)
        m = mix_summary.get(f'{metric}_mean', np.nan)
        rows.append({'metric': metric, 'baseline_mean': b, 'mix_mean': m, 'delta_mix_minus_base': m - b})
    display(pd.DataFrame(rows))

Let's now try alpha=0.75

In [ ]:
import numpy as np
import torch
from torch.utils.data import Dataset, ConcatDataset, WeightedRandomSampler, DataLoader
import pandas as pd

# Reuse synthetic dataset from previous section if available; otherwise load it.
if 'train_syn_ds' not in globals():
    SYN_SAVE_DIR = Path('/content/drive/MyDrive/sex_scp71_synthetic_balanced_female_mi3x_ckpt20000')
    syn_data = np.load(SYN_SAVE_DIR / 'syn_data_12.npy').astype(np.float32)
    syn_cond = np.load(SYN_SAVE_DIR / 'syn_cond_72.npy').astype(np.float32)

    if syn_cond.shape[1] == 2:
        syn_mi = syn_cond[:, 1].astype(np.float32)
    elif syn_cond.shape[1] >= 72:
        mi_cols = [1 + int(i) for i in mi_label_indices]
        syn_mi = (syn_cond[:, mi_cols].sum(axis=1) > 0).astype(np.float32)
    else:
        raise ValueError(f'Unexpected synthetic cond shape: {syn_cond.shape}')

    class SyntheticECGMIDataset(Dataset):
        def __init__(self, signals, mi_labels):
            self.signals = np.asarray(signals, dtype=np.float32)
            self.labels = np.asarray(mi_labels, dtype=np.float32)

        def __len__(self):
            return len(self.signals)

        def __getitem__(self, idx):
            return {
                'signal': torch.tensor(self.signals[idx], dtype=torch.float32),
                'label': torch.tensor(self.labels[idx], dtype=torch.float32),
            }

    train_syn_ds = SyntheticECGMIDataset(syn_data, syn_mi)


# Mixing config
ALPHA = 0.75
MIX_TRAIN_SEEDS = [0, 1, 2, 3, 4]
EPOCHS_MIX = 100
LR_MIX = 1e-3
VERBOSE_MIX_EPOCHS = False

R = len(train_real_ds)
S = len(train_syn_ds)
num_samples_per_epoch = R  # keep batches/epoch aligned with baseline
baseline_batches = (R + batch_size - 1) // batch_size

print('real train size:', R)
print('synthetic train size:', S)
print('alpha:', ALPHA)
print('num_samples_per_epoch:', num_samples_per_epoch)


def make_mix_train_loader(alpha: float, train_seed: int) -> DataLoader:
    eps = 1e-10
    a = float(np.clip(alpha, eps, 1.0 - eps))

    combined = ConcatDataset([train_real_ds, train_syn_ds])

    # Expected mass: a on real block, (1-a) on synthetic block
    w = np.concatenate(
        [
            np.full(R, (a / R), dtype=np.float64),
            np.full(S, ((1.0 - a) / S), dtype=np.float64),
        ]
    )

    g = torch.Generator()
    g.manual_seed(int(train_seed))

    sampler = WeightedRandomSampler(
        weights=torch.as_tensor(w, dtype=torch.double),
        num_samples=int(num_samples_per_epoch),
        replacement=False,  # requested: WITHOUT replacement
        generator=g,
    )

    loader = DataLoader(
        combined,
        batch_size=batch_size,
        sampler=sampler,
        num_workers=2,
        pin_memory=True,
    )

    if len(loader) != baseline_batches:
        raise RuntimeError(
            f'Batch count mismatch: mix len(loader)={len(loader)} vs baseline {baseline_batches}'
        )

    return loader


mix_results = []
xresnet_mix = None

for mix_seed in MIX_TRAIN_SEEDS:
    print(f"\n========== MIX seed={mix_seed} (alpha={ALPHA}) ==========")
    set_train_seed(mix_seed)

    train_mix_loader = make_mix_train_loader(ALPHA, mix_seed)

    xresnet_mix = XResNet1d50BinaryMIClassifier(in_channels=12).to(device)
    xresnet_mix = fit_model(
        xresnet_mix,
        train_mix_loader,
        val_loader,
        device,
        epochs=EPOCHS_MIX,
        lr=LR_MIX,
        verbose=VERBOSE_MIX_EPOCHS,
    )

    tm = evaluate(xresnet_mix, test_loader, device)
    fem = subgroup_metrics(tm['targets'], tm['probs'], female_test_mask)
    mal = subgroup_metrics(tm['targets'], tm['probs'], male_test_mask)

    row = {
        'alpha': float(ALPHA),
        'mix_seed': int(mix_seed),
        'epochs': int(EPOCHS_MIX),
        'samples_per_epoch': int(num_samples_per_epoch),
        'batches_per_epoch': int(baseline_batches),
        'test_loss': float(tm['loss']),
        'test_auroc': float(tm['auroc']),
        'test_auprc': float(tm['auprc']),
        'female_auroc': float(fem['auroc']),
        'female_auprc': float(fem['auprc']),
        'male_auroc': float(mal['auroc']),
        'male_auprc': float(mal['auprc']),
    }
    mix_results.append(row)

    print(
        f"TEST | AUROC={row['test_auroc']:.4f} AUPRC={row['test_auprc']:.4f} | "
        f"F_AUROC={row['female_auroc']:.4f} M_AUROC={row['male_auroc']:.4f}"
    )


mix_summary = {'alpha': float(ALPHA), 'n_seeds': len(mix_results)}
for key in ['test_auroc', 'test_auprc', 'female_auroc', 'female_auprc', 'male_auroc', 'male_auprc']:
    a = np.array([r[key] for r in mix_results], dtype=np.float64)
    mix_summary[f'{key}_mean'] = float(a.mean())
    mix_summary[f'{key}_std'] = float(a.std(ddof=1)) if len(a) > 1 else 0.0

print('\n========== MIX aggregate over seeds (alpha=0.5) ==========')
for key in ['test_auroc', 'test_auprc', 'female_auroc', 'female_auprc', 'male_auroc', 'male_auprc']:
    print(f"  {key}: {mix_summary[key + '_mean']:.4f} ± {mix_summary[key + '_std']:.4f}")

display(pd.DataFrame(mix_results))

if 'summary' in globals():
    print('\nBaseline (real-only) mean vs Mix(alpha=0.5) mean:')
    rows = []
    for metric in ['test_auroc', 'test_auprc', 'female_auroc', 'female_auprc', 'male_auroc', 'male_auprc']:
        b = summary.get(f'{metric}_mean', np.nan)
        m = mix_summary.get(f'{metric}_mean', np.nan)
        rows.append({'metric': metric, 'baseline_mean': b, 'mix_mean': m, 'delta_mix_minus_base': m - b})
    display(pd.DataFrame(rows))

## Baseline: ResNet + Multi-Head Attention (real-only)

This section trains the attention-based MI classifier on the same balanced real subset used above. Settings here are **100 epochs** and **5 seeds**.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F


class MyResidualBlock(nn.Module):
    def __init__(self, downsample):
        super().__init__()
        self.downsample = downsample
        self.stride = 2 if self.downsample else 1
        K = 9
        P = (K - 1) // 2
        self.conv1 = nn.Conv2d(
            in_channels=256,
            out_channels=256,
            kernel_size=(1, K),
            stride=(1, self.stride),
            padding=(0, P),
            bias=False,
        )
        self.bn1 = nn.BatchNorm2d(256)
        self.conv2 = nn.Conv2d(
            in_channels=256,
            out_channels=256,
            kernel_size=(1, K),
            padding=(0, P),
            bias=False,
        )
        self.bn2 = nn.BatchNorm2d(256)
        if self.downsample:
            self.idfunc_0 = nn.AvgPool2d(kernel_size=(1, 2), stride=(1, 2))
            self.idfunc_1 = nn.Conv2d(
                in_channels=256, out_channels=256, kernel_size=(1, 1), bias=False
            )

    def forward(self, x):
        identity = x
        x = F.leaky_relu(self.bn1(self.conv1(x)))
        x = F.leaky_relu(self.bn2(self.conv2(x)))
        if self.downsample:
            identity = self.idfunc_0(identity)
            identity = self.idfunc_1(identity)
            if identity.shape[-1] != x.shape[-1]:
                identity = F.interpolate(
                    identity,
                    size=(identity.shape[2], x.shape[-1]),
                    mode='nearest',
                )
        return x + identity


class ResnetAttention(nn.Module):
    """nOUT=1 for binary MI; forward returns logits for BCEWithLogitsLoss."""

    def __init__(self, nOUT=1):
        super().__init__()
        self.conv = nn.Conv2d(
            in_channels=12,
            out_channels=256,
            kernel_size=(1, 15),
            padding=(0, 7),
            stride=(1, 2),
            bias=False,
        )
        self.bn = nn.BatchNorm2d(256)
        self.rb_0 = MyResidualBlock(downsample=True)
        self.rb_1 = MyResidualBlock(downsample=True)
        self.rb_2 = MyResidualBlock(downsample=True)
        self.rb_3 = MyResidualBlock(downsample=True)
        self.rb_4 = MyResidualBlock(downsample=True)
        self.mha = nn.MultiheadAttention(256, 8)
        self.pool = nn.AdaptiveMaxPool1d(output_size=1)
        self.fc_1 = nn.Linear(256 + 12, nOUT)

    def forward(self, x, l):
        # x: (B, 12, 1, T), l: (B, 12)
        x = F.leaky_relu(self.bn(self.conv(x)))
        x = self.rb_0(x)
        x = self.rb_1(x)
        x = self.rb_2(x)
        x = self.rb_3(x)
        x = self.rb_4(x)
        x = F.dropout(x, p=0.5, training=self.training)
        x = x.squeeze(2).permute(2, 0, 1)
        x, _ = self.mha(x, x, x)
        x = x.permute(1, 2, 0)
        x = self.pool(x).squeeze(2)
        x = torch.cat((x, l), dim=1)
        x = self.fc_1(x)
        return x.squeeze(-1)


class ResnetAttentionBinaryMI(nn.Module):
    """ECG (B,12,T) -> logits (B,)."""

    def __init__(self):
        super().__init__()
        self.backbone = ResnetAttention(nOUT=1)

    def forward(self, signal):
        x = signal.unsqueeze(2)
        l = signal.mean(dim=-1)
        return self.backbone(x, l)


_g = torch.Generator().manual_seed(0)
_x = torch.randn(2, 12, 1000, generator=_g)
_m = ResnetAttentionBinaryMI()
assert _m(_x).shape == (2,), _m(_x).shape
print('ResnetAttentionBinaryMI I/O check OK (2, 12, 1000) -> (2,)')

In [ ]:
# Baseline: ResNet+Attention on real-only balanced subset
import numpy as np
import pandas as pd

ATTN_EPOCHS = 40
ATTN_LR = 1e-3
ATTN_VERBOSE = False
ATTN_TRAIN_SEEDS = [0, 1, 2, 3, 4]

attn_baseline_results = []
attn_baseline_model = None

for train_seed in ATTN_TRAIN_SEEDS:
    print(f"\n========== [ATTN] TRAIN_SEED {train_seed} ==========")
    set_train_seed(train_seed)

    train_real_loader = make_train_real_loader(train_seed)
    attn_baseline_model = ResnetAttentionBinaryMI().to(device)
    attn_baseline_model = fit_model(
        attn_baseline_model,
        train_real_loader,
        val_loader,
        device,
        epochs=ATTN_EPOCHS,
        lr=ATTN_LR,
        verbose=ATTN_VERBOSE,
    )

    tm = evaluate(attn_baseline_model, test_loader, device)
    fem = subgroup_metrics(tm['targets'], tm['probs'], female_test_mask)
    mal = subgroup_metrics(tm['targets'], tm['probs'], male_test_mask)

    print(
        f"[ATTN] TEST seed={train_seed} | AUROC={tm['auroc']:.4f} AUPRC={tm['auprc']:.4f} | "
        f"F_AUROC={fem['auroc']:.4f} M_AUROC={mal['auroc']:.4f}"
    )

    attn_baseline_results.append(
        {
            'train_seed': int(train_seed),
            'test_loss': float(tm['loss']),
            'test_auroc': float(tm['auroc']),
            'test_auprc': float(tm['auprc']),
            'female_auroc': float(fem['auroc']),
            'female_auprc': float(fem['auprc']),
            'male_auroc': float(mal['auroc']),
            'male_auprc': float(mal['auprc']),
        }
    )

attn_baseline_summary = {
    'model': 'ResnetAttentionBinaryMI',
    'epochs': int(ATTN_EPOCHS),
    'lr': float(ATTN_LR),
    'seeds': ATTN_TRAIN_SEEDS,
}
for key in ['test_auroc', 'test_auprc', 'female_auroc', 'female_auprc', 'male_auroc', 'male_auprc']:
    a = np.array([r[key] for r in attn_baseline_results], dtype=np.float64)
    attn_baseline_summary[f'{key}_mean'] = float(a.mean())
    attn_baseline_summary[f'{key}_std'] = float(a.std(ddof=1)) if len(a) > 1 else 0.0

print('\n========== [ATTN] BASELINE aggregate over seeds ==========')
for key in ['test_auroc', 'test_auprc', 'female_auroc', 'female_auprc', 'male_auroc', 'male_auprc']:
    print(f"  {key}: {attn_baseline_summary[key + '_mean']:.4f} ± {attn_baseline_summary[key + '_std']:.4f}")

display(pd.DataFrame(attn_baseline_results))

# Optional quick comparison with XResNet baseline if available
if 'summary' in globals():
    rows = []
    for metric in ['test_auroc', 'test_auprc', 'female_auroc', 'female_auprc', 'male_auroc', 'male_auprc']:
        x = summary.get(f'{metric}_mean', np.nan)
        a = attn_baseline_summary.get(f'{metric}_mean', np.nan)
        rows.append({'metric': metric, 'xresnet_mean': x, 'attn_mean': a, 'delta_attn_minus_xresnet': a - x})
    print('\nXResNet baseline vs Attention baseline (mean over seeds):')
    display(pd.DataFrame(rows))

# Mixing 

In [ ]:
# ResNetAttention Mixing (alpha = 0.5, without replacement)
# Same mixing protocol as XResNet: weighted sampling from concatenated
# real + synthetic train sets with alpha=0.5, replacement=False,
# and same samples-per-epoch as real-only baseline.


In [ ]:
import numpy as np
import torch
from pathlib import Path
from torch.utils.data import Dataset, ConcatDataset, WeightedRandomSampler, DataLoader
import pandas as pd

# Reuse synthetic dataset from earlier sections if present; otherwise load now.
if 'train_syn_ds' not in globals():
    SYN_SAVE_DIR = Path('/content/drive/MyDrive/sex_scp71_synthetic_balanced_female_mi3x_ckpt20000')
    syn_data = np.load(SYN_SAVE_DIR / 'syn_data_12.npy').astype(np.float32)
    syn_cond = np.load(SYN_SAVE_DIR / 'syn_cond_72.npy').astype(np.float32)

    if syn_cond.shape[1] == 2:
        syn_mi = syn_cond[:, 1].astype(np.float32)
    elif syn_cond.shape[1] >= 72:
        mi_cols = [1 + int(i) for i in mi_label_indices]
        syn_mi = (syn_cond[:, mi_cols].sum(axis=1) > 0).astype(np.float32)
    else:
        raise ValueError(f'Unexpected synthetic cond shape: {syn_cond.shape}')

    class SyntheticECGMIDataset(Dataset):
        def __init__(self, signals, mi_labels):
            self.signals = np.asarray(signals, dtype=np.float32)
            self.labels = np.asarray(mi_labels, dtype=np.float32)

        def __len__(self):
            return len(self.signals)

        def __getitem__(self, idx):
            return {
                'signal': torch.tensor(self.signals[idx], dtype=torch.float32),
                'label': torch.tensor(self.labels[idx], dtype=torch.float32),
            }

    train_syn_ds = SyntheticECGMIDataset(syn_data, syn_mi)

ATTN_MIX_ALPHA = 0.5
ATTN_MIX_EPOCHS = 40
ATTN_MIX_LR = 1e-3
ATTN_MIX_VERBOSE = False
ATTN_MIX_SEEDS = [0, 1, 2, 3, 4]

R_a = len(train_real_ds)
S_a = len(train_syn_ds)
num_samples_per_epoch_a = R_a
baseline_batches_a = (R_a + batch_size - 1) // batch_size

print('real train size:', R_a)
print('synthetic train size:', S_a)
print('alpha:', ATTN_MIX_ALPHA)
print('samples/epoch:', num_samples_per_epoch_a)


def make_mix_train_loader_attn(alpha: float, train_seed: int):
    eps = 1e-10
    a = float(np.clip(alpha, eps, 1.0 - eps))

    combined = ConcatDataset([train_real_ds, train_syn_ds])
    w = np.concatenate(
        [
            np.full(R_a, (a / R_a), dtype=np.float64),
            np.full(S_a, ((1.0 - a) / S_a), dtype=np.float64),
        ]
    )

    g = torch.Generator()
    g.manual_seed(int(train_seed))

    sampler = WeightedRandomSampler(
        weights=torch.as_tensor(w, dtype=torch.double),
        num_samples=int(num_samples_per_epoch_a),
        replacement=False,  # requested: without replacement
        generator=g,
    )

    loader = DataLoader(
        combined,
        batch_size=batch_size,
        sampler=sampler,
        num_workers=2,
        pin_memory=True,
    )

    if len(loader) != baseline_batches_a:
        raise RuntimeError(f'Batch mismatch: {len(loader)} vs {baseline_batches_a}')

    return loader


attn_mix_results = []
attn_mix_model = None

for mix_seed in ATTN_MIX_SEEDS:
    print(f"\n========== [ATTN MIX] seed={mix_seed} alpha={ATTN_MIX_ALPHA} ==========")
    set_train_seed(mix_seed)

    train_mix_loader = make_mix_train_loader_attn(ATTN_MIX_ALPHA, mix_seed)
    attn_mix_model = ResnetAttentionBinaryMI().to(device)

    attn_mix_model = fit_model(
        attn_mix_model,
        train_mix_loader,
        val_loader,
        device,
        epochs=ATTN_MIX_EPOCHS,
        lr=ATTN_MIX_LR,
        verbose=ATTN_MIX_VERBOSE,
    )

    tm = evaluate(attn_mix_model, test_loader, device)
    fem = subgroup_metrics(tm['targets'], tm['probs'], female_test_mask)
    mal = subgroup_metrics(tm['targets'], tm['probs'], male_test_mask)

    row = {
        'alpha': float(ATTN_MIX_ALPHA),
        'mix_seed': int(mix_seed),
        'epochs': int(ATTN_MIX_EPOCHS),
        'samples_per_epoch': int(num_samples_per_epoch_a),
        'batches_per_epoch': int(baseline_batches_a),
        'test_loss': float(tm['loss']),
        'test_auroc': float(tm['auroc']),
        'test_auprc': float(tm['auprc']),
        'female_auroc': float(fem['auroc']),
        'female_auprc': float(fem['auprc']),
        'male_auroc': float(mal['auroc']),
        'male_auprc': float(mal['auprc']),
    }
    attn_mix_results.append(row)

    print(
        f"[ATTN MIX] TEST | AUROC={row['test_auroc']:.4f} AUPRC={row['test_auprc']:.4f} | "
        f"F_AUROC={row['female_auroc']:.4f} M_AUROC={row['male_auroc']:.4f}"
    )


attn_mix_summary = {'alpha': float(ATTN_MIX_ALPHA), 'n_seeds': len(attn_mix_results)}
for key in ['test_auroc', 'test_auprc', 'female_auroc', 'female_auprc', 'male_auroc', 'male_auprc']:
    a = np.array([r[key] for r in attn_mix_results], dtype=np.float64)
    attn_mix_summary[f'{key}_mean'] = float(a.mean())
    attn_mix_summary[f'{key}_std'] = float(a.std(ddof=1)) if len(a) > 1 else 0.0

print('\n========== [ATTN MIX] aggregate over seeds (alpha=0.5) ==========')
for key in ['test_auroc', 'test_auprc', 'female_auroc', 'female_auprc', 'male_auroc', 'male_auprc']:
    print(f"  {key}: {attn_mix_summary[key + '_mean']:.4f} ± {attn_mix_summary[key + '_std']:.4f}")

display(pd.DataFrame(attn_mix_results))

if 'attn_baseline_summary' in globals():
    rows = []
    for metric in ['test_auroc', 'test_auprc', 'female_auroc', 'female_auprc', 'male_auroc', 'male_auprc']:
        b = attn_baseline_summary.get(f'{metric}_mean', np.nan)
        m = attn_mix_summary.get(f'{metric}_mean', np.nan)
        rows.append({'metric': metric, 'attn_baseline_mean': b, 'attn_mix_mean': m, 'delta_mix_minus_baseline': m - b})
    print('\nAttention baseline vs attention mix (mean over seeds):')
    display(pd.DataFrame(rows))

We now try alpha 0.75

In [ ]:
import numpy as np
import torch
from pathlib import Path
from torch.utils.data import Dataset, ConcatDataset, WeightedRandomSampler, DataLoader
import pandas as pd

# Reuse synthetic dataset from earlier sections if present; otherwise load now.
if 'train_syn_ds' not in globals():
    SYN_SAVE_DIR = Path('/content/drive/MyDrive/sex_scp71_synthetic_balanced_female_mi3x_ckpt20000')
    syn_data = np.load(SYN_SAVE_DIR / 'syn_data_12.npy').astype(np.float32)
    syn_cond = np.load(SYN_SAVE_DIR / 'syn_cond_72.npy').astype(np.float32)

    if syn_cond.shape[1] == 2:
        syn_mi = syn_cond[:, 1].astype(np.float32)
    elif syn_cond.shape[1] >= 72:
        mi_cols = [1 + int(i) for i in mi_label_indices]
        syn_mi = (syn_cond[:, mi_cols].sum(axis=1) > 0).astype(np.float32)
    else:
        raise ValueError(f'Unexpected synthetic cond shape: {syn_cond.shape}')

    class SyntheticECGMIDataset(Dataset):
        def __init__(self, signals, mi_labels):
            self.signals = np.asarray(signals, dtype=np.float32)
            self.labels = np.asarray(mi_labels, dtype=np.float32)

        def __len__(self):
            return len(self.signals)

        def __getitem__(self, idx):
            return {
                'signal': torch.tensor(self.signals[idx], dtype=torch.float32),
                'label': torch.tensor(self.labels[idx], dtype=torch.float32),
            }

    train_syn_ds = SyntheticECGMIDataset(syn_data, syn_mi)

ATTN_MIX_ALPHA = 0.75
ATTN_MIX_EPOCHS = 40
ATTN_MIX_LR = 1e-3
ATTN_MIX_VERBOSE = False
ATTN_MIX_SEEDS = [0, 1, 2, 3, 4]

R_a = len(train_real_ds)
S_a = len(train_syn_ds)
num_samples_per_epoch_a = R_a
baseline_batches_a = (R_a + batch_size - 1) // batch_size

print('real train size:', R_a)
print('synthetic train size:', S_a)
print('alpha:', ATTN_MIX_ALPHA)
print('samples/epoch:', num_samples_per_epoch_a)


def make_mix_train_loader_attn(alpha: float, train_seed: int):
    eps = 1e-10
    a = float(np.clip(alpha, eps, 1.0 - eps))

    combined = ConcatDataset([train_real_ds, train_syn_ds])
    w = np.concatenate(
        [
            np.full(R_a, (a / R_a), dtype=np.float64),
            np.full(S_a, ((1.0 - a) / S_a), dtype=np.float64),
        ]
    )

    g = torch.Generator()
    g.manual_seed(int(train_seed))

    sampler = WeightedRandomSampler(
        weights=torch.as_tensor(w, dtype=torch.double),
        num_samples=int(num_samples_per_epoch_a),
        replacement=False,  # requested: without replacement
        generator=g,
    )

    loader = DataLoader(
        combined,
        batch_size=batch_size,
        sampler=sampler,
        num_workers=2,
        pin_memory=True,
    )

    if len(loader) != baseline_batches_a:
        raise RuntimeError(f'Batch mismatch: {len(loader)} vs {baseline_batches_a}')

    return loader


attn_mix_results = []
attn_mix_model = None

for mix_seed in ATTN_MIX_SEEDS:
    print(f"\n========== [ATTN MIX] seed={mix_seed} alpha={ATTN_MIX_ALPHA} ==========")
    set_train_seed(mix_seed)

    train_mix_loader = make_mix_train_loader_attn(ATTN_MIX_ALPHA, mix_seed)
    attn_mix_model = ResnetAttentionBinaryMI().to(device)

    attn_mix_model = fit_model(
        attn_mix_model,
        train_mix_loader,
        val_loader,
        device,
        epochs=ATTN_MIX_EPOCHS,
        lr=ATTN_MIX_LR,
        verbose=ATTN_MIX_VERBOSE,
    )

    tm = evaluate(attn_mix_model, test_loader, device)
    fem = subgroup_metrics(tm['targets'], tm['probs'], female_test_mask)
    mal = subgroup_metrics(tm['targets'], tm['probs'], male_test_mask)

    row = {
        'alpha': float(ATTN_MIX_ALPHA),
        'mix_seed': int(mix_seed),
        'epochs': int(ATTN_MIX_EPOCHS),
        'samples_per_epoch': int(num_samples_per_epoch_a),
        'batches_per_epoch': int(baseline_batches_a),
        'test_loss': float(tm['loss']),
        'test_auroc': float(tm['auroc']),
        'test_auprc': float(tm['auprc']),
        'female_auroc': float(fem['auroc']),
        'female_auprc': float(fem['auprc']),
        'male_auroc': float(mal['auroc']),
        'male_auprc': float(mal['auprc']),
    }
    attn_mix_results.append(row)

    print(
        f"[ATTN MIX] TEST | AUROC={row['test_auroc']:.4f} AUPRC={row['test_auprc']:.4f} | "
        f"F_AUROC={row['female_auroc']:.4f} M_AUROC={row['male_auroc']:.4f}"
    )


attn_mix_summary = {'alpha': float(ATTN_MIX_ALPHA), 'n_seeds': len(attn_mix_results)}
for key in ['test_auroc', 'test_auprc', 'female_auroc', 'female_auprc', 'male_auroc', 'male_auprc']:
    a = np.array([r[key] for r in attn_mix_results], dtype=np.float64)
    attn_mix_summary[f'{key}_mean'] = float(a.mean())
    attn_mix_summary[f'{key}_std'] = float(a.std(ddof=1)) if len(a) > 1 else 0.0

print('\n========== [ATTN MIX] aggregate over seeds (alpha=0.5) ==========')
for key in ['test_auroc', 'test_auprc', 'female_auroc', 'female_auprc', 'male_auroc', 'male_auprc']:
    print(f"  {key}: {attn_mix_summary[key + '_mean']:.4f} ± {attn_mix_summary[key + '_std']:.4f}")

display(pd.DataFrame(attn_mix_results))

if 'attn_baseline_summary' in globals():
    rows = []
    for metric in ['test_auroc', 'test_auprc', 'female_auroc', 'female_auprc', 'male_auroc', 'male_auprc']:
        b = attn_baseline_summary.get(f'{metric}_mean', np.nan)
        m = attn_mix_summary.get(f'{metric}_mean', np.nan)
        rows.append({'metric': metric, 'attn_baseline_mean': b, 'attn_mix_mean': m, 'delta_mix_minus_baseline': m - b})
    print('\nAttention baseline vs attention mix (mean over seeds):')
    display(pd.DataFrame(rows))

# Performance on synthetic only

In [ ]:
# Synthetic-only XResNet baseline (train on synthetic, eval on real val/test)
import numpy as np
import torch
from pathlib import Path
from torch.utils.data import Dataset, DataLoader
import pandas as pd

# --- synthetic source ---
SYN_SAVE_DIR = Path('/content/drive/MyDrive/sex_scp71_synthetic_balanced_female_mi3x_ckpt20000')
syn_data_path = SYN_SAVE_DIR / 'syn_data_12.npy'
syn_cond_path = SYN_SAVE_DIR / 'syn_cond_72.npy'

if not syn_data_path.is_file() or not syn_cond_path.is_file():
    raise FileNotFoundError(f"Missing synthetic files in {SYN_SAVE_DIR}")

syn_data = np.load(syn_data_path).astype(np.float32)
syn_cond = np.load(syn_cond_path).astype(np.float32)

# Derive MI labels robustly from cond format
# cond can be [sex, mi] or [sex + 71 labels]
if syn_cond.shape[1] == 2:
    syn_mi = syn_cond[:, 1].astype(np.float32)
elif syn_cond.shape[1] >= 72:
    mi_cols = [1 + int(i) for i in mi_label_indices]  # offset by sex channel
    syn_mi = (syn_cond[:, mi_cols].sum(axis=1) > 0).astype(np.float32)
else:
    raise ValueError(f"Unexpected syn_cond shape: {syn_cond.shape}")

class SyntheticECGMIDataset(Dataset):
    def __init__(self, signals, labels):
        self.signals = np.asarray(signals, dtype=np.float32)
        self.labels = np.asarray(labels, dtype=np.float32)

    def __len__(self):
        return len(self.signals)

    def __getitem__(self, idx):
        return {
            "signal": torch.tensor(self.signals[idx], dtype=torch.float32),
            "label": torch.tensor(self.labels[idx], dtype=torch.float32),
        }

train_syn_only_ds = SyntheticECGMIDataset(syn_data, syn_mi)

def make_train_syn_only_loader(train_seed: int) -> DataLoader:
    g = torch.Generator()
    g.manual_seed(int(train_seed))
    return DataLoader(
        train_syn_only_ds,
        batch_size=batch_size,
        shuffle=True,
        num_workers=2,
        pin_memory=True,
        generator=g,
    )

print("Synthetic-only train size:", len(train_syn_only_ds))
print("Synthetic MI counts:", np.unique(syn_mi, return_counts=True))
print("Real val/test sizes:", len(val_real_ds), len(test_real_ds))

# --- training config ---
SYN_ONLY_SEEDS = [0, 1, 2, 3, 4]
SYN_ONLY_EPOCHS = 75
SYN_ONLY_LR = 1e-3
SYN_ONLY_VERBOSE = False

syn_only_results = []

for train_seed in SYN_ONLY_SEEDS:
    print(f"\n========== SYN-ONLY TRAIN_SEED {train_seed} ==========")
    set_train_seed(train_seed)

    train_loader_syn = make_train_syn_only_loader(train_seed)
    model_syn = XResNet1d50BinaryMIClassifier(in_channels=12).to(device)

    model_syn = fit_model(
        model_syn,
        train_loader_syn,
        val_loader,           # real validation
        device,
        epochs=SYN_ONLY_EPOCHS,
        lr=SYN_ONLY_LR,
        verbose=SYN_ONLY_VERBOSE,
    )

    tm = evaluate(model_syn, test_loader, device)   # real test
    fem = subgroup_metrics(tm["targets"], tm["probs"], female_test_mask)
    mal = subgroup_metrics(tm["targets"], tm["probs"], male_test_mask)

    print(
        f"SYN-ONLY TEST seed={train_seed} | AUROC={tm['auroc']:.4f} AUPRC={tm['auprc']:.4f} | "
        f"F_AUROC={fem['auroc']:.4f} M_AUROC={mal['auroc']:.4f}"
    )

    syn_only_results.append(
        {
            "train_seed": int(train_seed),
            "test_loss": float(tm["loss"]),
            "test_auroc": float(tm["auroc"]),
            "test_auprc": float(tm["auprc"]),
            "female_auroc": float(fem["auroc"]),
            "female_auprc": float(fem["auprc"]),
            "male_auroc": float(mal["auroc"]),
            "male_auprc": float(mal["auprc"]),
        }
    )

syn_only_summary = {
    "setup": "xresnet synthetic-only train, real val/test",
    "epochs": int(SYN_ONLY_EPOCHS),
    "lr": float(SYN_ONLY_LR),
    "seeds": SYN_ONLY_SEEDS,
}
for key in ["test_auroc", "test_auprc", "female_auroc", "female_auprc", "male_auroc", "male_auprc"]:
    a = np.array([r[key] for r in syn_only_results], dtype=np.float64)
    syn_only_summary[f"{key}_mean"] = float(a.mean())
    syn_only_summary[f"{key}_std"] = float(a.std(ddof=1)) if len(a) > 1 else 0.0

print("\n========== SYN-ONLY aggregate over seeds ==========")
for key in ["test_auroc", "test_auprc", "female_auroc", "female_auprc", "male_auroc", "male_auprc"]:
    print(f"  {key}: {syn_only_summary[key + '_mean']:.4f} ± {syn_only_summary[key + '_std']:.4f}")

display(pd.DataFrame(syn_only_results))

# Optional: compare with real-only baseline summary if available
if "summary" in globals():
    rows = []
    for metric in ["test_auroc", "test_auprc", "female_auroc", "female_auprc", "male_auroc", "male_auprc"]:
        b = summary.get(f"{metric}_mean", np.nan)
        s = syn_only_summary.get(f"{metric}_mean", np.nan)
        rows.append({"metric": metric, "real_only_mean": b, "syn_only_mean": s, "delta_syn_minus_real": s - b})
    print("\nReal-only vs Synthetic-only (mean over seeds):")
    display(pd.DataFrame(rows))